In [1]:
%pip install datashader colorcet duckdb

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import duckdb
import datashader as ds
import datashader.transfer_functions as tf
import colorcet as cc
from PIL import Image

# --- 1. SETUP ---
PATH_GFW = "/mnt/shared_data/finflow/gfw_raw/*/*.parquet"
PATH_OBIS = "/mnt/shared_data/finflow/obis_raw/*/*/*.parquet"
OUTPUT_DIR = "/mnt/shared_data/finflow/images/tiles"

ZOOM = 7
NUM_TILES = 2**ZOOM
TILE_SIZE = 256

# Global Bounds
X_MIN, X_MAX = -180, 180
Y_MIN, Y_MAX = -90, 90

x_step = (X_MAX - X_MIN) / NUM_TILES
y_step = (Y_MAX - Y_MIN) / NUM_TILES

# Connect to DuckDB
con = duckdb.connect()

# --- 2. THE LOOP ---
print(f"Starting Zoom {ZOOM} generation ({NUM_TILES}x{NUM_TILES} tiles)...")

for x_idx in range(NUM_TILES):
    # Calculate longitude boundaries for this vertical column
    x0 = X_MIN + (x_idx * x_step)
    x1 = x0 + x_step
    
    # Create directory for this X column
    column_dir = os.path.join(OUTPUT_DIR, str(ZOOM), str(x_idx))
    os.makedirs(column_dir, exist_ok=True)
    
    # LOAD ONLY THIS COLUMN'S DATA (Corrected Syntax)
    query_gfw = f"SELECT lon, lat, hours FROM read_parquet('{PATH_GFW}') WHERE lon BETWEEN {x0} AND {x1}"
    query_obis = f"SELECT decimalLongitude as lon, decimalLatitude as lat FROM read_parquet('{PATH_OBIS}') WHERE decimalLongitude BETWEEN {x0} AND {x1}"
    
    # .execute() returns a result object, then .df() converts to DataFrame
    df_gfw_col = con.execute(query_gfw).df()
    df_obis_col = con.execute(query_obis).df()
    
    # Process every tile in this column
    for y_idx in range(NUM_TILES):
        y_top = Y_MAX - (y_idx * y_step)
        y_bottom = y_top - y_step
        
        cvs = ds.Canvas(plot_width=TILE_SIZE, plot_height=TILE_SIZE, 
                        x_range=(x0, x1), y_range=(y_bottom, y_top))
        
        # Aggregate (handling empty dataframes to avoid errors)
        agg_gfw = cvs.points(df_gfw_col, 'lon', 'lat', ds.sum('hours')) if not df_gfw_col.empty else None
        agg_obis = cvs.points(df_obis_col, 'lon', 'lat', ds.count()) if not df_obis_col.empty else None
        
        # Shade
        img_gfw = tf.shade(agg_gfw, cmap=cc.fire, how='log') if agg_gfw is not None else None
        img_obis = tf.shade(agg_obis, cmap=["#90ee90", "#00ff00"], how='log') if agg_obis is not None else None
        
        # Combine/Save
        if img_gfw is not None or img_obis is not None:
            # Use tf.stack to layer them, or create a blank tile if one is missing
            layers = [l for l in [img_gfw, img_obis] if l is not None]
            tile_img = tf.stack(*layers).to_pil().convert("RGBA")
        else:
            # Entirely empty tile (transparent)
            tile_img = Image.new("RGBA", (TILE_SIZE, TILE_SIZE), (0, 0, 0, 0))
            
        tile_img.save(os.path.join(column_dir, f"{y_idx}.png"))
        
    print(f"Finished Column {x_idx}/{NUM_TILES} | GFW Points in this sliver: {len(df_gfw_col)}")

print("Done! Check your tiles folder.")

Starting Zoom 7 generation (128x128 tiles)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Finished Column 0/128 | GFW Points in this sliver: 1534379


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Finished Column 1/128 | GFW Points in this sliver: 1524212


...

Finished Column 125/128 | GFW Points in this sliver: 2210076


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Finished Column 126/128 | GFW Points in this sliver: 2375092


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Finished Column 127/128 | GFW Points in this sliver: 1908869
Done! Check your tiles folder.
